# 어디살지 - 카카오 좌표·주변 인프라 수집 (+ NEIS 학원 동 단위, 선택)

**하는 일**
- 학교 12곳과 샘플 단지 46곳의 **실제 좌표**를 찾습니다.
- 학교 주변의 지하철역·병원·약국·대형마트·어린이집/유치원·문화시설·공원 **실제 이름과 거리**, 그리고 반경 500m/1km/2km 내 **학원 수**를 모읍니다.
- (선택) NEIS 키를 넣으면 학원·교습소 전체 목록으로 **동 단위** 집계도 만듭니다.

**준비물**: 카카오 개발자 콘솔의 **REST API 키** (필수), NEIS 인증키 (선택)

**사용법 (클릭 4번)**
1. 위쪽 메뉴 **런타임 → 모두 실행**을 누릅니다.
2. 첫 번째 입력 칸에 카카오 **REST API 키**를 붙여넣고 Enter. 두 번째 칸은 NEIS 키 (없으면 그냥 Enter).
3. 2~5분 기다립니다.
4. `kakao_neis_result.json` 이 자동으로 내려받아집니다. 이 파일을 Claude 채팅에 첨부해 주세요.

오류가 나면 빨간 글씨 설명 문구만 복사해서 알려주세요. (키는 빼고)

In [ ]:
# -*- coding: utf-8 -*-
"""어디살지: 카카오 로컬 API 로 좌표·주변 인프라 수집 + (선택) NEIS 학원 전체 목록으로 동 단위 집계.
외부 패키지 불필요. run_kakao(KAKAO_KEY, SCHOOLS, APTS) / run_neis(NEIS_KEY) 를 호출."""
import json, math, time, urllib.parse, urllib.request
from collections import Counter

KAKAO = "https://dapi.kakao.com/v2/local/"
CATS = {"SW8": ("지하철역", 1500), "HP8": ("병원", 1000), "PM9": ("약국", 1000), "MT1": ("대형마트", 1500), "PS3": ("어린이집·유치원", 800), "CT1": ("문화시설", 2000)}
HINT = {401: "카카오 REST API 키가 올바르지 않아요. 앱 키 메뉴의 'REST API 키'를 복사했는지 확인하세요.",
        403: "카카오맵(로컬) 사용 설정이 꺼져 있을 수 있어요. 카카오 개발자 콘솔 > 내 애플리케이션 > 제품 설정 에서 카카오맵(로컬)을 활성화하세요.",
        429: "호출 한도를 넘었어요. 잠시 후 다시 실행하세요."}

class KakaoError(Exception): pass
def kget(key, path, **q):
    req = urllib.request.Request(KAKAO + path + "?" + urllib.parse.urlencode(q), headers={"Authorization": "KakaoAK " + key.strip()})
    for t in range(3):
        try: return json.load(urllib.request.urlopen(req, timeout=30))
        except urllib.error.HTTPError as e:
            if e.code in (401, 403): raise KakaoError(f"[{e.code}] {HINT[e.code]} (응답: {e.read()[:200].decode('utf8','ignore')})")
            if e.code == 429: time.sleep(2 + t); continue
            raise KakaoError(f"[{e.code}] {e.read()[:200].decode('utf8','ignore')}")
        except Exception as e: time.sleep(1 + t); last = e
    raise KakaoError(f"요청 실패: {last}")

def hav(x1, y1, x2, y2):
    r = 6371000; p1, p2 = math.radians(y1), math.radians(y2); dp = p2 - p1; dl = math.radians(x2 - x1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

def geocode_school(key, name, addr):
    d = kget(key, "search/address.json", query=addr)
    if d["documents"]: x = d["documents"][0]; return {"x": float(x["x"]), "y": float(x["y"]), "addr": addr, "by": "address"}
    d = kget(key, "search/keyword.json", query=name)
    if d["documents"]: x = d["documents"][0]; return {"x": float(x["x"]), "y": float(x["y"]), "addr": x.get("road_address_name") or x["address_name"], "by": "keyword"}
    return None

def geocode_apt(key, gu, name, dong):
    for q in (f"서울 {gu} {dong} {name}", f"서울 {gu} {name}", f"서울 {gu} {name}아파트"):
        d = kget(key, "search/keyword.json", query=q, size=5)
        for x in d["documents"]:
            if gu in x["address_name"] and (x.get("category_group_code") in ("", None, "AT4") or "아파트" in x.get("category_name", "") or "주거" in x.get("category_name", "")):
                return {"x": float(x["x"]), "y": float(x["y"]), "addr": x.get("road_address_name") or x["address_name"], "place": x["place_name"], "by": "keyword"}
    return None

def infra(key, x, y):
    out = {}
    for code, (label, rad) in CATS.items():
        d = kget(key, "search/category.json", category_group_code=code, x=x, y=y, radius=rad, sort="distance", size=5)
        out[code] = {"label": label, "radius": rad, "total": d["meta"]["total_count"], "items": [{"name": i["place_name"], "dist": int(i["distance"]), "x": float(i["x"]), "y": float(i["y"])} for i in d["documents"]]}
    d = kget(key, "search/keyword.json", query="공원", x=x, y=y, radius=1500, sort="distance", size=5)
    out["PARK"] = {"label": "공원", "radius": 1500, "total": d["meta"]["total_count"], "items": [{"name": i["place_name"], "dist": int(i["distance"]), "x": float(i["x"]), "y": float(i["y"])} for i in d["documents"]]}
    acad = {}
    for r in (500, 1000, 2000):
        acad[str(r)] = kget(key, "search/category.json", category_group_code="AC5", x=x, y=y, radius=r, size=1)["meta"]["total_count"]
    out["AC5"] = {"label": "학원", "counts": acad}
    return out

def run_kakao(key, schools, apts, log=print):
    """schools: {id:{name,addr}}  apts: [{id,gu,name,dong?,school}]"""
    res = {"schools": {}, "apts": {}, "failed": []}
    for sid, s in schools.items():
        g = geocode_school(key, s["name"], s["addr"])
        if not g: res["failed"].append("학교:" + sid); log("좌표 실패", s["name"]); continue
        g["infra"] = infra(key, g["x"], g["y"]); res["schools"][sid] = g; log("학교 완료", s["name"], "학원(1km)", g["infra"]["AC5"]["counts"]["1000"])
    for a in apts:
        g = geocode_apt(key, a["gu"], a["name"], a.get("dong", ""))
        if not g: res["failed"].append("단지:" + a["id"]); continue
        sc = res["schools"].get(a["school"])
        if sc:
            m = hav(g["x"], g["y"], sc["x"], sc["y"]); g["straight_m"] = int(m); g["walk_min_est"] = max(1, round(m * 1.3 / 80))
        res["apts"][a["id"]] = g
    log(f"단지 좌표 {len(res['apts'])}/{len(apts)} 완료")
    return res

# ---------- NEIS (선택): 키가 있으면 학원 전체 목록 -> 동 단위 집계 ----------
import re
def run_neis(key, gus, targets, log=print):
    out = {}
    for gu in gus:
        rows, page = [], 1
        while True:
            q = urllib.parse.urlencode({"KEY": key, "Type": "json", "pIndex": page, "pSize": 1000, "ATPT_OFCDC_SC_CODE": "B10", "ADMST_ZONE_NM": gu})
            d = json.load(urllib.request.urlopen("https://open.neis.go.kr/hub/acaInsTiInfo?" + q, timeout=60))
            if "acaInsTiInfo" not in d: break
            got = d["acaInsTiInfo"][1]["row"]; rows += got
            if len(got) < 1000: break
            page += 1
        op = [r for r in rows if r.get("REG_STTUS_NM") == "개원"]
        dongs = {}
        for r in op:
            m = re.search(r"\(([가-힣0-9]+동)", r.get("FA_RDNDA") or "")
            if m and m.group(1) in targets.get(gu, []):
                x = dongs.setdefault(m.group(1), {"total": 0, "boseup": 0, "hakwon": 0, "gyoseupso": 0, "capacity": 0})
                x["total"] += 1; x["boseup"] += r.get("REALM_SC_NM") == "입시.검정 및 보습"
                x["hakwon" if r["ACA_INSTI_SC_NM"] == "학원" else "gyoseupso"] += 1; x["capacity"] += int(r.get("TOFOR_SMTOT") or 0)
        out[gu] = {"registered": len(rows), "open": len(op), "dongs": dongs}; log(gu, "학원", len(rows), "건 중 개원", len(op))
    return out


SCHOOLS = {"hwimun": {"name": "휘문고등학교", "addr": "서울특별시 강남구 역삼로 541"}, "sookmyung": {"name": "숙명여자고등학교", "addr": "서울특별시 강남구 남부순환로 2807"}, "sehwa": {"name": "세화고등학교", "addr": "서울특별시 서초구 신반포로 56-7"}, "banpo": {"name": "반포고등학교", "addr": "서울특별시 서초구 고무래로 71"}, "jamsil": {"name": "잠실고등학교", "addr": "서울특별시 송파구 올림픽로33길 71"}, "boin": {"name": "보인고등학교", "addr": "서울특별시 송파구 오금로49길 18"}, "jinmyeong": {"name": "진명여자고등학교", "addr": "서울특별시 양천구 목동로 170"}, "yangjeong": {"name": "양정고등학교", "addr": "서울특별시 양천구 안양천로 1039"}, "seorabeol": {"name": "서라벌고등학교", "addr": "서울특별시 노원구 한글비석로5길 18"}, "cheongwon": {"name": "청원여자고등학교", "addr": "서울특별시 노원구 한글비석로 506"}, "baejae": {"name": "배재고등학교", "addr": "서울특별시 강동구 고덕로 227"}, "hanyoung": {"name": "한영고등학교", "addr": "서울특별시 강동구 동남로 832"}}
APTS = [{"id":"hwimun-0","gu":"강남구","name":"래미안대치팰리스","school":"hwimun"},{"id":"hwimun-1","gu":"강남구","name":"대치아이파크","school":"hwimun"},{"id":"hwimun-2","gu":"강남구","name":"도곡렉슬","school":"hwimun"},{"id":"hwimun-3","gu":"강남구","name":"역삼자이","school":"hwimun"},{"id":"sookmyung-0","gu":"강남구","name":"은마","school":"sookmyung"},{"id":"sookmyung-1","gu":"강남구","name":"한보미도맨션1","school":"sookmyung"},{"id":"sookmyung-2","gu":"강남구","name":"쌍용대치2","school":"sookmyung"},{"id":"sookmyung-3","gu":"강남구","name":"개포래미안포레스트","school":"sookmyung"},{"id":"sehwa-0","gu":"서초구","name":"반포자이","school":"sehwa"},{"id":"sehwa-1","gu":"서초구","name":"아크로리버파크","school":"sehwa"},{"id":"sehwa-2","gu":"서초구","name":"래미안퍼스티지","school":"sehwa"},{"id":"sehwa-3","gu":"서초구","name":"반포래미안아이파크","school":"sehwa"},{"id":"banpo-0","gu":"서초구","name":"래미안원베일리","school":"banpo"},{"id":"banpo-1","gu":"서초구","name":"반포센트럴자이","school":"banpo"},{"id":"banpo-2","gu":"서초구","name":"신반포자이","school":"banpo"},{"id":"jamsil-0","gu":"송파구","name":"리센츠","school":"jamsil"},{"id":"jamsil-1","gu":"송파구","name":"잠실엘스","school":"jamsil"},{"id":"jamsil-2","gu":"송파구","name":"트리지움","school":"jamsil"},{"id":"jamsil-3","gu":"송파구","name":"잠실주공5단지","school":"jamsil"},{"id":"boin-0","gu":"송파구","name":"헬리오시티","school":"boin"},{"id":"boin-1","gu":"송파구","name":"파크리오","school":"boin"},{"id":"boin-2","gu":"송파구","name":"올림픽선수기자촌1단지","school":"boin"},{"id":"boin-3","gu":"송파구","name":"잠실래미안아이파크","school":"boin"},{"id":"jinmyeong-0","gu":"양천구","name":"목동신시가지 5단지","school":"jinmyeong"},{"id":"jinmyeong-1","gu":"양천구","name":"목동신시가지 6단지","school":"jinmyeong"},{"id":"jinmyeong-2","gu":"양천구","name":"목동신시가지 7단지","school":"jinmyeong"},{"id":"jinmyeong-3","gu":"양천구","name":"목동2삼성래미안","school":"jinmyeong"},{"id":"yangjeong-0","gu":"양천구","name":"목동신시가지 11단지","school":"yangjeong"},{"id":"yangjeong-1","gu":"양천구","name":"목동신시가지 12단지","school":"yangjeong"},{"id":"yangjeong-2","gu":"양천구","name":"목동센트럴아이파크위브2단지","school":"yangjeong"},{"id":"yangjeong-3","gu":"양천구","name":"현대하이페리온","school":"yangjeong"},{"id":"seorabeol-0","gu":"노원구","name":"중계주공5단지","school":"seorabeol"},{"id":"seorabeol-1","gu":"노원구","name":"중계그린1단지","school":"seorabeol"},{"id":"seorabeol-2","gu":"노원구","name":"건영2","school":"seorabeol"},{"id":"seorabeol-3","gu":"노원구","name":"중계무지개","school":"seorabeol"},{"id":"cheongwon-0","gu":"노원구","name":"청구","school":"cheongwon"},{"id":"cheongwon-1","gu":"노원구","name":"롯데우성","school":"cheongwon"},{"id":"cheongwon-2","gu":"노원구","name":"불암현대","school":"cheongwon"},{"id":"cheongwon-3","gu":"노원구","name":"상계주공10단지","school":"cheongwon"},{"id":"baejae-0","gu":"강동구","name":"고덕그라시움","school":"baejae"},{"id":"baejae-1","gu":"강동구","name":"고덕아르테온","school":"baejae"},{"id":"baejae-2","gu":"강동구","name":"래미안힐스테이트고덕","school":"baejae"},{"id":"hanyoung-0","gu":"강동구","name":"고덕센트럴IPARK","school":"hanyoung"},{"id":"hanyoung-1","gu":"강동구","name":"고덕자이","school":"hanyoung"},{"id":"hanyoung-2","gu":"강동구","name":"올림픽파크포레온","school":"hanyoung"},{"id":"hanyoung-3","gu":"강동구","name":"고덕리엔파크3단지","school":"hanyoung"}]


In [ ]:
from getpass import getpass
import json
KAKAO_KEY = getpass('카카오 REST API 키를 붙여넣고 Enter: ')
NEIS_KEY = getpass('NEIS 인증키 (없으면 그냥 Enter): ').strip()
print('1) 카카오 연결 시험 중...')
kget(KAKAO_KEY, 'search/address.json', query='서울 강남구 역삼로 541')
print('2) 학교·단지 좌표와 주변 인프라 수집 중...')
result = run_kakao(KAKAO_KEY, SCHOOLS, APTS)
if NEIS_KEY:
    print('3) NEIS 학원 동 단위 집계 중...')
    TARGETS = {'강남구': ['대치동','도곡동','개포동','역삼동'], '서초구': ['반포동','잠원동'], '송파구': ['잠실동','신천동','방이동'], '양천구': ['목동'], '노원구': ['중계동','상계동'], '강동구': ['고덕동','상일동']}
    result['neis_academy'] = run_neis(NEIS_KEY, list(TARGETS), TARGETS)
json.dump(result, open('kakao_neis_result.json', 'w', encoding='utf-8'), ensure_ascii=False)
print('학교 좌표', len(result['schools']), '/ 12, 단지 좌표', len(result['apts']), '/ 46, 실패', result['failed'])
from google.colab import files
files.download('kakao_neis_result.json')
